# Pre-Processing for logs that are not in .xes.gz format

Converts CSV sources (and a BPIC12 smoke-test sample) to `.xes.gz` under `data/real/`. Also unpacks re-downloaded 4TU archives from `data/temp/` into `data/real/<LOG>/original_log.xes.gz` for published-source verification.

Run with the project root as the working directory, or from `analyses/` (paths resolve via `project_root`).

To verify that this notebook reproduces the working logs without overwriting them, set `OUTPUT_INFIX` (e.g. `".verify"`) so outputs become `NAME.verify.xes.gz`, then run the CSV comparison cell. Published XES logs are checked separately by comparing `original_log.xes.gz` to the working file (gzip MD5 and uncompressed XES MD5).


## Setup

In [ ]:
import hashlib
import gzip
import os
import zipfile
from pathlib import Path

import pandas as pd
from pm4py.objects.log.util import dataframe_utils
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.exporter.xes import exporter as xes_exporter

# Resolve project root whether cwd is the repo root or analyses/.
cwd = Path.cwd().resolve()
if (cwd / "data" / "real").is_dir():
    project_root = cwd
elif (cwd.parent / "data" / "real").is_dir():
    project_root = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate data/real from cwd={cwd}")

def data_path(*parts: str) -> Path:
    return project_root.joinpath("data", *parts)

# Inserted before ".xes.gz". Empty string writes the working filenames.
# Example: OUTPUT_INFIX = ".verify" -> ACCRE.verify.xes.gz
OUTPUT_INFIX = ".verify"

def xes_out(dataset_dir: str, stem: str) -> Path:
    """Build output path: data/real/<dataset_dir>/<stem><OUTPUT_INFIX>.xes.gz"""
    return data_path("real", dataset_dir, f"{stem}{OUTPUT_INFIX}.xes.gz")

def md5_bytes(data: bytes) -> str:
    return hashlib.md5(data).hexdigest()

def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def md5_gunzip(path: Path) -> str:
    h = hashlib.md5()
    with gzip.open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("project_root =", project_root)
print("OUTPUT_INFIX =", repr(OUTPUT_INFIX))


## Test log

Create a **500**-trace sample from BPIC 2012 to test the scripts (not to be used in formal evaluations).


In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.log.obj import EventLog

log_path = data_path("real", "BPIC12", "BPI_Challenge_2012.xes.gz")
out_path = xes_out("TEST_BPIC12", "TEST_BPI_Challenge_2012")

log = xes_importer.apply(str(log_path))
number_traces = 500
sampled_log = EventLog(log[:number_traces])
os.makedirs(out_path.parent, exist_ok=True)
xes_exporter.apply(sampled_log, str(out_path))
print("Wrote", out_path, "with", number_traces, "traces")


## ACCRE-log

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "ACCRE", "AcademicCredentials_train.csv.gz")
df = pd.read_csv(log_path, compression="gzip")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["start_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ACCRE", "ACCRE")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## BPIC14

In [ ]:
log_path = data_path("real", "BPIC14", "Detail_Incident_Activity.csv.gz")
df = pd.read_csv(log_path, delimiter=";", compression="gzip")

df["case:concept:name"] = df["Incident ID"]
df["concept:name"] = df["IncidentActivity_Type"]
df["time:timestamp"] = df["DateStamp"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("BPIC14", "BPIC14")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## CALL

The log has been split into test and train. Each contains only completed traces. Hence, some traces that start in the train data and finish in the test data have been removed. Because we want to avoid detecting drifts due to these removed traces, we only include the train dataset.


In [ ]:
log_path = data_path("real", "CALL", "CallCenter_train.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["case_id"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end_time"]
df["org:resource"] = df["resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("CALL", "CALL")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## ITHD

In [ ]:
log_path = data_path("real", "ITHD", "finale.csv.gz")
df = pd.read_csv(log_path, delimiter=",")

df["case:concept:name"] = df["Case ID"]
df["concept:name"] = df["Activity"]
df["time:timestamp"] = df["Complete Timestamp"]
df["org:resource"] = df["Resource"]

df = dataframe_utils.convert_timestamp_columns_in_df(df)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("ITHD", "ITHD")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## MOBIS

In [ ]:
log_path = data_path("real", "MOBIS", "10258_mobis_challenge_log_2019.csv.gz")
df = pd.read_csv(log_path, delimiter=";")

df["case:concept:name"] = df["case"]
df["concept:name"] = df["activity"]
df["time:timestamp"] = df["end"]

df["time:timestamp"] = pd.to_datetime(
    df["time:timestamp"],
    format="%d.%m.%Y %H:%M",
    utc=True,
)
df = df.sort_values(by="time:timestamp").reset_index(drop=True)

event_log = log_converter.apply(df, variant=log_converter.Variants.TO_EVENT_LOG)
output_path = xes_out("MOBIS", "MOBIS")
xes_exporter.apply(event_log, str(output_path), parameters={"gzip": True})
print("Wrote", output_path, "| events:", len(df), "| traces:", df["case:concept:name"].nunique())


## Compare regenerated outputs to working logs

Compares each `stem{OUTPUT_INFIX}.xes.gz` against the working `stem.xes.gz` in the same folder.

- **gzip_identical**: byte-identical compressed files
- **raw_xes_identical**: identical after gunzip (ignores gzip wrapper/metadata)
- If PM4Py re-export changes XML formatting, also report trace/event counts from both files.


In [ ]:
from pm4py.objects.log.importer.xes import importer as xes_importer

pairs = [
    ("ACCRE", "ACCRE.xes.gz"),
    ("BPIC14", "BPIC14.xes.gz"),
    ("CALL", "CALL.xes.gz"),
    ("ITHD", "ITHD.xes.gz"),
    ("MOBIS", "MOBIS.xes.gz"),
    ("TEST_BPIC12", "TEST_BPI_Challenge_2012.xes.gz"),
]

if not OUTPUT_INFIX:
    print("OUTPUT_INFIX is empty — skipping comparison (would compare a file to itself).")
else:
    rows = []
    for dataset, working_name in pairs:
        working = data_path("real", dataset, working_name)
        stem = working_name[: -len(".xes.gz")]
        regenerated = data_path("real", dataset, f"{stem}{OUTPUT_INFIX}.xes.gz")
        if not regenerated.is_file():
            rows.append((dataset, "MISSING_REGEN", "-", "-", "-", "-"))
            continue
        if not working.is_file():
            rows.append((dataset, "MISSING_WORKING", "-", "-", "-", "-"))
            continue
        gz_eq = md5_file(working) == md5_file(regenerated)
        raw_eq = md5_gunzip(working) == md5_gunzip(regenerated)
        if raw_eq:
            n_w = n_r = "n/a (raw match)"
        else:
            wlog = xes_importer.apply(str(working))
            rlog = xes_importer.apply(str(regenerated))
            n_w = f"{len(wlog)} traces / {sum(len(t) for t in wlog)} events"
            n_r = f"{len(rlog)} traces / {sum(len(t) for t in rlog)} events"
        rows.append((dataset, "ok", gz_eq, raw_eq, n_w, n_r))

    print(f"{'dataset':12} {'status':16} {'gz_eq':6} {'raw_eq':6} working_counts | regenerated_counts")
    for dataset, status, gz_eq, raw_eq, n_w, n_r in rows:
        print(f"{dataset:12} {status:16} {str(gz_eq):6} {str(raw_eq):6} {n_w} | {n_r}")


## Unpack published 4TU downloads (`data/temp` → `original_log.xes.gz`)

Place each re-download from [4TU.ResearchData](https://data.4tu.nl) under `data/temp/` (zip archives as published, or a raw `.xes` for BPIC19). Running the next cell writes `data/real/<LOG>/original_log.xes.gz`:

- zip containing `.xes.gz` → extract that member as `original_log.xes.gz`
- zip containing uncompressed `.xes` (BPIC15) → gzip the `.xes` member into `original_log.xes.gz`
- raw uncompressed `.xes` (BPIC19) → gzip into `original_log.xes.gz`

Set `OVERWRITE_ORIGINALS = False` to skip folders that already have `original_log.xes.gz`.

In [ ]:
OVERWRITE_ORIGINALS = False
temp_dir = data_path("temp")

# (dataset_dir, temp filename, zip member or None if temp file is already the log)
PUBLISHED_DOWNLOADS = [
    # BPIC15: zip holds uncompressed .xes (+ duplicate DATA1.xml)
    ("BPIC15_1", "BPI Challenge 2015 Municipality 1_1_all (1).zip", "BPIC15_1.xes"),
    ("BPIC15_2", "BPI Challenge 2015 Municipality 2_1_all (1).zip", "BPIC15_2.xes"),
    ("BPIC15_3", "BPI Challenge 2015 Municipality 3_1_all (1).zip", "BPIC15_3.xes"),
    ("BPIC15_4", "BPI Challenge 2015 Municipality 4_1_all (1).zip", "BPIC15_4.xes"),
    ("BPIC15_5", "BPI Challenge 2015 Municipality 5_1_all (1).zip", "BPIC15_5.xes"),
    # Already gzipped XES inside zip
    ("BPIC17", "BPI Challenge 2017_1_all.zip", "BPI Challenge 2017.xes.gz"),
    ("BPIC18", "BPI Challenge 2018_1_all (1).zip", "BPI Challenge 2018.xes.gz"),
    ("BPIC20_dd", "BPI Challenge 2020_ Domestic Declarations_1_all.zip", "DomesticDeclarations.xes.gz"),
    ("BPIC20_id", "BPI Challenge 2020_ International Declarations_1_all (1).zip", "InternationalDeclarations.xes.gz"),
    ("BPIC20_ptc", "BPI Challenge 2020_ Prepaid Travel Costs_1_all (1).zip", "PrepaidTravelCost.xes.gz"),
    ("BPIC20_rfp", "BPI Challenge 2020_ Request For Payment_1_all (1).zip", "RequestForPayment.xes.gz"),
    ("BPIC20_tpd", "BPI Challenge 2020_ Travel Permit Data_1_all (1).zip", "PermitLog.xes.gz"),
    ("RTFMP", "Road Traffic Fine Management Process_1_all.zip", "Road_Traffic_Fine_Management_Process.xes.gz"),
    ("SEPSIS", "Sepsis Cases - Event Log_1_all.zip", "Sepsis Cases - Event Log.xes.gz"),
    # BPIC19: raw uncompressed .xes (not zipped)
    ("BPIC19", "BPI_Challenge_2019 (1).xes", None),
]

def write_original_from_zip(zip_path: Path, member: str, dest: Path) -> None:
    with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(dest, "wb") as out:
        if member.endswith(".gz"):
            out.write(src.read())
        else:
            with gzip.GzipFile(fileobj=out, mode="wb", filename=Path(member).name) as gz:
                while True:
                    chunk = src.read(1 << 20)
                    if not chunk:
                        break
                    gz.write(chunk)

def write_original_from_xes(xes_path: Path, dest: Path) -> None:
    with open(xes_path, "rb") as src, gzip.open(dest, "wb") as out:
        while True:
            chunk = src.read(1 << 20)
            if not chunk:
                break
            out.write(chunk)

for dataset, temp_name, member in PUBLISHED_DOWNLOADS:
    src = temp_dir / temp_name
    dest = data_path("real", dataset, "original_log.xes.gz")
    if not src.is_file():
        print(f"SKIP {dataset}: missing {src}")
        continue
    if dest.is_file() and not OVERWRITE_ORIGINALS:
        print(f"SKIP {dataset}: {dest.name} exists (set OVERWRITE_ORIGINALS=True to replace)")
        continue
    dest.parent.mkdir(parents=True, exist_ok=True)
    if member is None:
        write_original_from_xes(src, dest)
        print(f"Wrote {dest} (gzip of {temp_name})")
    else:
        write_original_from_zip(src, member, dest)
        how = "extract" if member.endswith(".gz") else "gzip"
        print(f"Wrote {dest} ({how} {member})")

## Compare published originals to working logs

For each dataset below, compares `original_log.xes.gz` (re-download) to the working `.xes.gz`:

- **gzip_identical**: byte-identical compressed files
- **raw_xes_identical**: identical after gunzip (gzip wrapper/metadata may differ)

In [ ]:
published_pairs = [
    ("BPIC11", "hospital_log.xes.gz"),
    ("BPIC12", "BPI_Challenge_2012.xes.gz"),
    ("BPIC13_cp", "BPI_Challenge_2013_closed_problems.xes.gz"),
    ("BPIC13_i", "BPI_Challenge_2013_incidents.xes.gz"),
    ("BPIC13_op", "BPI_Challenge_2013_open_problems.xes.gz"),
    ("BPIC15_1", "BPIC15_1.xes.gz"),
    ("BPIC15_2", "BPIC15_2.xes.gz"),
    ("BPIC15_3", "BPIC15_3.xes.gz"),
    ("BPIC15_4", "BPIC15_4.xes.gz"),
    ("BPIC15_5", "BPIC15_5.xes.gz"),
    ("BPIC17", "BPI Challenge 2017.xes.gz"),
    ("BPIC18", "BPI Challenge 2018.xes.gz"),
    ("BPIC19", "BPI_Challenge_2019.xes.gz"),
    ("BPIC20_dd", "DomesticDeclarations.xes.gz"),
    ("BPIC20_id", "InternationalDeclarations.xes.gz"),
    ("BPIC20_ptc", "PrepaidTravelCost.xes.gz"),
    ("BPIC20_rfp", "RequestForPayment.xes.gz"),
    ("BPIC20_tpd", "PermitLog.xes.gz"),
    ("RTFMP", "Road_Traffic_Fine_Management_Process.xes.gz"),
    ("SEPSIS", "Sepsis Cases - Event Log.xes.gz"),
]

rows = []
for dataset, working_name in published_pairs:
    working = data_path("real", dataset, working_name)
    original = data_path("real", dataset, "original_log.xes.gz")
    if not original.is_file():
        rows.append((dataset, "MISSING_ORIGINAL", "-", "-"))
        continue
    if not working.is_file():
        rows.append((dataset, "MISSING_WORKING", "-", "-"))
        continue
    gz_eq = md5_file(working) == md5_file(original)
    raw_eq = md5_gunzip(working) == md5_gunzip(original)
    status = "exact" if gz_eq else ("raw_match" if raw_eq else "MISMATCH")
    rows.append((dataset, status, gz_eq, raw_eq))

print(f"{'dataset':12} {'status':16} {'gz_eq':6} {'raw_eq':6}")
for dataset, status, gz_eq, raw_eq in rows:
    print(f"{dataset:12} {status:16} {str(gz_eq):6} {str(raw_eq):6}")